In [ ]:
import pypsa

import matplotlib.pyplot as plt

In [ ]:
from _helpers_notebooks import mock_snakemake

snakemake = mock_snakemake(
    "collect_figures",
    scenario="default",  # default, penalty-sa, penalty-ea, penalty-nwa, penalty-oc
)

In [ ]:
config = snakemake.config

In [ ]:
colors = config["colors"]

### Read networks in Network collection

In [ ]:
nc = {}

for scenario in config["scenario"].keys():
    trade_model_fn = f"../../results/cost_year~2030/interone~hbi/intertwo~eaf-grid/final~steel/scenario~{scenario}/network.nc"

    n = pypsa.Network(trade_model_fn)
    n.name = scenario
    nc[scenario] = n

nc = pypsa.NetworkCollection(list(nc.values()))

### Trade volume

In [ ]:
fig, ax, facet_col = nc.statistics.transmission.plot.bar()

### Costs

In [ ]:
df = nc.statistics.system_cost(groupby="carrier").to_frame("systemcost")
steel_demand = (
    nc.statistics.withdrawal().to_frame().loc["Load", "default", "steel"].value
)

df = df / steel_demand
# df

In [ ]:
# Pivot to get networks as rows and carriers as columns
plot_df = df.pivot_table(
    index="network", columns="carrier", values="systemcost", aggfunc="sum"
)

order = ["iron_ore", "shipping_iron_ore", "hbi", "shipping_hbi", "steel"]

plot_df = plot_df[order]

plot_df.rename(
    columns={
        "iron_ore": "Iron ore",
        "shipping_iron_ore": "Shipping iron ore",
        "hbi": "DRI",
        "shipping_hbi": "Shipping HBI",
        "steel": "EAF",
    },
    inplace=True,
)

In [ ]:
# Create plot
ax = plot_df.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 4),
    color=[nc["default"].carriers.color[c] for c in order],
    alpha=0.8,
)

# ---- Axis labels ----
ax.set_ylabel("Total cost in €/t$_{steel}$")
ax.set_xlabel("Scenario")

# ---- Title ----
# ax.set_title("Cost of Steel")

# ---- Y-axis grid ----
ax.yaxis.grid(True, linestyle="-", alpha=0.6)
ax.set_axisbelow(True)

# ---- Add total values on top of bars ----
totals = plot_df.sum(axis=1)

for i, total in enumerate(totals):
    ax.text(
        i,
        total,
        f"{total:.1f}",  # adjust scaling if needed
        ha="center",
        va="bottom",
        fontsize=10,
    )

# ---- Flip legend order ----
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[::-1],
    labels[::-1],
    title="Cost component",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)

plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(snakemake.input.cost_comparison, dpi=300)
plt.savefig(snakemake.input.cost_comparison_png, dpi=300)
plt.show()

In [ ]:
nc.statistics.market_value()

### Experimental area

In [ ]:
nc["default"].statistics.capex().div(1e6)

In [ ]:
nc["default"].statistics.opex().div(1e6)

In [ ]:
nc.statistics.capex().div(1e6)

In [ ]:
nc.statistics.energy_balance.plot.bar()